In [19]:
!pip install mysql-connector-python
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import mysql.connector
from mysql.connector import Error
import os


def read_parquet_as_df(parquet_path: str):
    return pq.read_table(parquet_path).to_pandas()

#### Show Parquet as DF

In [5]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\etl_output\PV.EST\PV.EST.parquet")
df

,country,year,value
0,Aruba,1960,NaN
1,Aruba,1961,NaN
2,Aruba,1962,NaN
3,Aruba,1963,NaN
4,Aruba,1964,NaN
...,...,...,...
17019,Zimbabwe,2019,-0.943303
17020,Zimbabwe,2020,-1.052743
17021,Zimbabwe,2021,-0.954443
17022,Zimbabwe,2022,-0.894974


#### Shrinking

In [18]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\etl_output\SP.POP.TOTL\SP.POP.TOTL.parquet")
output_path = r"/workspace/qgi\qgi-4.0\data\etl_output\SP.POP.TOTL_shrinked\SP.POP.TOTL_shrinked.parquet"

# Select 5 countries to keep
target_countries = ["United States", "United Kingdom", "Germany", "France", "Canada", "Australia", "Japan", "India", "China", "Brazil", "Mexico", "South Africa", "Nigeria", "Egypt", "Turkiye", "Russia", "Italy", "Spain", "Argentina", "South Korea", "Indonesia", "Pakistan", "Bangladesh", "Thailand", "Philippines", "Vietnam", "Kenya", "Tanzania", "Uganda", "Ghana", "Senegal", "Morocco", "Algeria", "Israel", "Saudi Arabia", "Iran", "Iraq", "Afghanistan", "Poland", "Hungary", "Czech", "Greece", "Netherlands", "Belgium", "Sweden", "Norway", "Denmark", "Finland", "Switzerland", "Austria"]

# Filter the DataFrame
filtered_df = df[df["country"].isin(target_countries)]

filtered_df.to_parquet(output_path, index = False, compression = "snappy")
print(f"Shrinked Parquet file saved to {output_path}")
filtered_df

Shrinked Parquet file saved to /workspace/qgi\qgi-4.0\data\etl_output\SP.POP.TOTL_shrinked\SP.POP.TOTL_shrinked.parquet


,country,year,value
128,Afghanistan,1960,9035043.0
129,Afghanistan,1961,9214083.0
130,Afghanistan,1962,9404406.0
131,Afghanistan,1963,9604487.0
132,Afghanistan,1964,9814318.0
...,...,...,...
16891,South Africa,2019,59587885.0
16892,South Africa,2020,60562381.0
16893,South Africa,2021,61502603.0
16894,South Africa,2022,62378410.0


#### Calculating Comaprisons

In [11]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\etl_output\PV.EST_shrinked\PV.EST_shrinked.parquet")


def preprocess_data(df):
    """
    Preprocesses data for each country by handling NaN values and organizing data for analysis.
    
    This function processes the input DataFrame to prepare data for each country:
    1. Filters the data for each country
    2. Sorts the data by year
    3. Handles NaN values by either skipping countries with more than one NaN or interpolating single NaN values
    4. Creates a list of dictionaries containing the country name, original data, and cleaned values
    
    Parameters:
    df (pd.DataFrame): The input DataFrame containing indicator data for multiple countries
    
    Returns:
    list: A list of dictionaries, where each dictionary contains:
        - "country": The country name
        - "data": The original country data sorted by year
        - "values": Cleaned numeric values with NaNs handled
    """
    countries = df["country"].unique()
    preprocessed_data = []

    for country in countries:
        country_data = df[df["country"] == country].sort_values("year")
        values = country_data["value"].values.copy()

        # Handle NaN values
        nan_indices = np.isnan(values)
        if np.sum(nan_indices) > 1:
            continue

        if np.sum(nan_indices) == 1:
            nan_pos = np.where(nan_indices)[0][0]
            if nan_pos == 0:
                values[nan_pos] = values[nan_pos + 1]
            elif nan_pos == len(values) - 1:
                values[nan_pos] = values[nan_pos - 1]
            else:
                values[nan_pos] = (values[nan_pos - 1] + values[nan_pos + 1]) / 2

        preprocessed_data.append({
            "country": country,
            "data": country_data,
            "values": values
        })

    return preprocessed_data

def calculate_total_comparisons(df, min_length=3, max_length=5):
    """
    Calculate the total number of segment comparisons for progress tracking.
    
    This function computes the total number of comparisons that would be made 
    during the pattern extraction process. It considers all possible segment 
    lengths within the specified range and all country pairs.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame containing indicator data for multiple countries
    min_length (int): Minimum segment length to consider
    max_length (int): Maximum segment length to consider
    
    Returns:
    int: Total number of comparisons that would be performed
    """
    preprocessed_data = preprocess_data(df)
    total_comparisons = 0

    for i, country_a in enumerate(preprocessed_data):
        values_a = country_a["values"]

        for segment_length in range(min_length, max_length + 1):
            if len(values_a) < segment_length:
                continue

            num_segments_a = len(values_a) - segment_length + 1
            num_segments_b = sum(
                len(country_b["values"]) - segment_length + 1
                for country_b in preprocessed_data[i+1:]
                if len(country_b["values"]) >= segment_length
            )
            total_comparisons += num_segments_a * num_segments_b

    return total_comparisons

# Example usage:
total = calculate_total_comparisons(df)
print(f"Total comparisons: {total}")

Total comparisons: 0


#### Size Saving Comparison

In [14]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\patternizer_output\SP.POP.TOTL_shrinked\SP.POP.TOTL_shrinked_sdcis.parquet")
print(df.head())

   country_a_id  country_b_id  length  start_year_a  start_year_b  correlation
0            45            81       3          1960          1977     0.945475
1            45            81       3          1961          1960     0.970550
2            45            81       3          1961          1961     0.964687
3            45            81       3          1961          1962     0.948679
4            45            81       3          1961          1963     0.965124


In [15]:
df

,country_a_id,country_b_id,length,start_year_a,start_year_b,correlation
0,45,81,3,1960,1977,0.945475
1,45,81,3,1961,1960,0.970550
2,45,81,3,1961,1961,0.964687
3,45,81,3,1961,1962,0.948679
4,45,81,3,1961,1963,0.965124
...,...,...,...,...,...,...
82764,110,236,5,2005,2015,0.958875
82765,110,236,5,2005,2017,0.943587
82766,110,236,5,2005,2018,0.978710
82767,110,236,5,2005,2019,0.901636


In [13]:
df

,country_a,country_b,length,start_year_a,start_year_b,correlation,data_segment_a,data_segment_b
0,China,Germany,3,1960,1977,0.945475,"[667070000.0, 660330000.0, 665770000.0]","[78159814.0, 78091820.0, 78126350.0]"
1,China,Germany,3,1961,1960,0.970550,"[660330000.0, 665770000.0, 682335000.0]","[72814900.0, 73377632.0, 74025784.0]"
2,China,Germany,3,1961,1961,0.964687,"[660330000.0, 665770000.0, 682335000.0]","[73377632.0, 74025784.0, 74714353.0]"
3,China,Germany,3,1961,1962,0.948679,"[660330000.0, 665770000.0, 682335000.0]","[74025784.0, 74714353.0, 75318337.0]"
4,China,Germany,3,1961,1963,0.965124,"[660330000.0, 665770000.0, 682335000.0]","[74714353.0, 75318337.0, 75963695.0]"
...,...,...,...,...,...,...,...,...
82764,Japan,United States,5,2005,2015,0.958875,"[127773000.0, 127854000.0, 128001000.0, 128063...","[320738994.0, 323071755.0, 325122128.0, 326838..."
82765,Japan,United States,5,2005,2017,0.943587,"[127773000.0, 127854000.0, 128001000.0, 128063...","[325122128.0, 326838199.0, 328329953.0, 331526..."
82766,Japan,United States,5,2005,2018,0.978710,"[127773000.0, 127854000.0, 128001000.0, 128063...","[326838199.0, 328329953.0, 331526933.0, 332048..."
82767,Japan,United States,5,2005,2019,0.901636,"[127773000.0, 127854000.0, 128001000.0, 128063...","[328329953.0, 331526933.0, 332048977.0, 333271..."


#### Comparing Parquet with MySQL

##### MySQL Setup

In [ ]:
# def create_mysql_user():
#     try:
#         # Connect as root or another admin user
#         connection = mysql.connector.connect(
#             host='localhost',
#             user='root',
#             password='<MYSQL_ROOT_PASSWORD>'
#         )

#         if connection.is_connected():
#             cursor = connection.cursor()
#             cursor.execute("CREATE USER 'qgi_user'@'localhost' IDENTIFIED BY '<MYSQL_PASSWORD>")
#             cursor.execute("GRANT ALL PRIVILEGES ON sdci_pattern_db.* TO 'qgi_user'@'localhost';")
#             cursor.execute("FLUSH PRIVILEGES;")
#             connection.commit()
#             print("MySQL user created successfully")

#     except Error as e:
#         print(f"Error creating MySQL user: {e}")
#     finally:
#         if connection.is_connected():
#             cursor.close()
#             connection.close()

# create_mysql_user()

MySQL user created successfully


In [ ]:
# def grant_privileges_to_user():
#     try:
#         # Connect as root or another admin user
#         admin_connection = mysql.connector.connect(
#             host='localhost',
#             user='root',  # Or another admin username
#             password='<MYSQL_ROOT_PASSWORD>'
#         )

#         if admin_connection.is_connected():
#             admin_cursor = admin_connection.cursor()
            
#             # Grant necessary privileges to 'amin' user
#             admin_cursor.execute("GRANT CREATE ON *.* TO 'qgi_user'@'localhost';")
#             admin_cursor.execute("GRANT DROP ON *.* TO 'qgi_user'@'localhost';")
#             admin_cursor.execute("FLUSH PRIVILEGES;")
            
#             # Commit the changes
#             admin_connection.commit()
#             print("Privileges granted successfully to user 'amin'")

#     except Error as e:
#         print(f"Error granting privileges: {e}")
#     finally:
#         if admin_connection.is_connected():
#             admin_cursor.close()
#             admin_connection.close()
#             print("Admin MySQL connection is closed")

# # Run the function to grant privileges
# grant_privileges_to_user()

Privileges granted successfully to user 'amin'
Admin MySQL connection is closed


In [ ]:
# def create_database():
#     """Create a new MySQL database and required tables"""
#     try:
#         # Connect to MySQL server
#         connection = mysql.connector.connect(
#             host='localhost',
#             user='amin',
#             password='<MYSQL_PASSWORD>'
#         )
        
#         if connection.is_connected():
#             db_info = connection.get_server_info()
#             print(f"Connected to MySQL Server version {db_info}")
            
#             cursor = connection.cursor()
            
#             # Create database
#             db_name = "sdci_database"
#             cursor.execute(f"CREATE DATABASE IF NOT EXISTS {db_name}")
#             print(f"Database '{db_name}' created successfully")
            
#             # Use the new database
#             cursor.execute(f"USE {db_name}")
            
#             # Create SDCIs table
#             create_table_query = """
#             CREATE TABLE IF NOT EXISTS sdcis (
#                 id INT AUTO_INCREMENT PRIMARY KEY,
#                 country_a_id INT NOT NULL,
#                 country_b_id INT NOT NULL,
#                 length INT NOT NULL,
#                 start_year_a INT NOT NULL,
#                 start_year_b INT NOT NULL,
#                 correlation FLOAT NOT NULL,
#                 indicator_id VARCHAR(255) NOT NULL,
#                 created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
#             ) ENGINE=InnoDB;
#             """
#             cursor.execute(create_table_query)
#             print("Table 'sdcis' created successfully")
            
#             # Commit the transaction
#             connection.commit()

#     except Error as e:
#         print(f"Error while creating database: {e}")
#     finally:
#         if connection.is_connected():
#             cursor.close()
#             connection.close()
#             print("MySQL connection is closed")

# create_database()

Connected to MySQL Server version 8.0.41
Database 'sdci_database' created successfully
Table 'sdcis' created successfully
MySQL connection is closed


##### Comparison

In [20]:


def get_parquet_file_size(file_path):
    """Get the size of a Parquet file in MB"""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file {file_path} does not exist")
    return os.path.getsize(file_path) / (1024 * 1024)  # Convert to MB

def get_mysql_table_size(host, user, password, database, table_name):
    """Get the size of a MySQL table in MB"""
    try:
        connection = mysql.connector.connect(
            host=host,
            user=user,
            password=password,
            database=database
        )
        cursor = connection.cursor()
        
        query = f"""
        SELECT 
            TABLE_NAME, 
            ROUND(SUM(DATA_LENGTH + INDEX_LENGTH) / 1024 / 1024, 2) AS table_size_mb 
        FROM 
            information_schema.TABLES 
        WHERE 
            TABLE_SCHEMA = '{database}' 
            AND TABLE_NAME = '{table_name}'
        GROUP BY 
            TABLE_NAME;
        """
        cursor.execute(query)
        result = cursor.fetchone()
        if result:
            return result[1]
        else:
            return 0
        
    except Error as e:
        print(f"Error: {e}")
        return None
    finally:
        if connection.is_connected():
            cursor.close()
            connection.close()

# Example usage
parquet_file_path = r"/workspace/qgi\qgi-4.0\data\patternizer_output\SP.POP.TOTL\SP.POP.TOTL_sdcis.parquet"
mysql_host = "localhost"
mysql_user = "amin"
mysql_password = "<MYSQL_PASSWORD>"
mysql_database = "sdci_database"
mysql_table = "sdcis"

# Get Parquet file size
try:
    parquet_size = get_parquet_file_size(parquet_file_path)
    print(f"Parquet file size: {parquet_size:.2f} MB")
except FileNotFoundError as e:
    print(e)

# Get MySQL table size
mysql_size = get_mysql_table_size(
    mysql_host, 
    mysql_user, 
    mysql_password, 
    mysql_database, 
    mysql_table
)
if mysql_size is not None:
    print(f"MySQL table size: {mysql_size} MB")

Parquet file size: 21.08 MB
Error: 1045 (28000): Access denied for user 'qgi_user'@'localhost' (using password: YES)


UnboundLocalError: cannot access local variable 'connection' where it is not associated with a value